* Multiple linear regression: the same ŷ = x @ w + b as Day 6, but x has many features, one weight per feature.

* One-hot encoding: ocean_proximity is text. pd.get_dummies(df, columns=["ocean_proximity"], dtype=int) turns it into 0/1 columns, one per category (like your is_female). It is safe before the split, because it uses only the category names, not numbers learned from the data.

* Polynomial regression: PolynomialFeatures(degree=2, include_bias=False) turns x into the columns [x, x²]. The model ŷ = w₁x + w₂x² + b draws a curve in x, but it is still linear regression on the new columns.

* StandardScaler: the z-score from Day 5 applied to each column (mean 0, std 1), with the mean and std learned from training data only. Inside a Pipeline, this is automatic. Here it keeps large powers of x manageable.

* Overfitting: a model that is too flexible memorises the training set. Its train score rises while its test score falls. Judge by the test score, never the train score.

* pd.get_dummies(df, columns=[...], dtype=int) → a new DataFrame with 0/1 columns.
* PolynomialFeatures(degree, include_bias=False) → a step that adds powers of each feature.
* StandardScaler() → a step that rescales each column using the training mean and std.
* pipe.score(X, y) → R² for regression (for classification it was accuracy).

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

url = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"
df = pd.read_csv(url)
df = df[df["median_house_value"] < 500001].copy()       # remove the capped rows
print(df.shape)

(19675, 10)


In [4]:
def report(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{name:12s} MAE={mae:,.0f}  RMSE={rmse:,.0f}  R2={r2:.4f}")

In [5]:
# 1. one feature (median_income), baseline vs linear
X1 = df[["median_income"]]
y = df["median_house_value"]
X_train, X_test, y_train, y_test = train_test_split(
    X1, y, test_size=0.2, random_state=42
)
print(X_train.shape, X_test.shape)
report("Baseline:", y_test, DummyRegressor(strategy="mean").fit(X_train, y_train).predict(X_test))
m1 = LinearRegression().fit(X_train, y_train)
report("Linear 1:", y_test, m1.predict(X_test))
print(round(m1.coef_[0], 2), round(m1.intercept_, 2))

(15740, 1) (3935, 1)
Baseline:    MAE=80,135  RMSE=99,865  R2=-0.0002
Linear 1:    MAE=57,911  RMSE=76,059  R2=0.4198
39789.25 46223.84


In [9]:
# 2. all features
d = pd.get_dummies(df, columns=["ocean_proximity"], dtype=int)
print(d.shape)
X = d.drop(columns="median_house_value")
y2 = d["median_house_value"]
Xa_train, Xa_test, ya_train, ya_test = train_test_split(
    X, y2, test_size=0.2, random_state=42
)
print(Xa_train["total_bedrooms"].isna().sum(), Xa_test["total_bedrooms"].isna().sum())
pipe_all = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", LinearRegression()),
])
pipe_all.fit(Xa_train, ya_train)
report("Linear All:", ya_test, pipe_all.predict(Xa_test))
print(round(pipe_all.score(Xa_train, ya_train), 4), round(pipe_all.score(Xa_test, ya_test)))
coefs = pd.Series(pipe_all.named_steps["model"].coef_, index=X.columns)
print(coefs.sort_values().round(0).head(3).to_dict())
print(coefs.sort_values().round(0).tail(3).to_dict())                  

(19675, 14)
162 38
Linear All:  MAE=45,917  RMSE=62,562  R2=0.6075
0.6128 1
{'ocean_proximity_INLAND': -67660.0, 'ocean_proximity_NEAR BAY': -36585.0, 'ocean_proximity_<1H OCEAN': -28014.0}
{'housing_median_age': 913.0, 'median_income': 37332.0, 'ocean_proximity_ISLAND': 156722.0}


In [11]:
# 3. polynomial regression on median_income (full training data)
def poly_pipe(degree):
    return Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("scale", StandardScaler()),
        ("model", LinearRegression())
    ])
for degree in (1, 2, 3):
    p = poly_pipe(degree).fit(X_train, y_train)
    print("Full:", degree, round(p.score(X_train, y_train), 4), round(p.score(X_test, y_test), 4))

Full: 1 0.4111 0.4198
Full: 2 0.4116 0.4204
Full: 3 0.4158 0.4238


In [12]:
# 4. overfitting: train on only 30 rows
X_small, y_small = X_train.iloc[:30], y_train.iloc[:30]
for degree in (1, 2, 3):
    p = poly_pipe(degree).fit(X_small, y_small)
    print("Small:", degree, round(p.score(X_small, y_small), 4), round(p.score(X_test, y_test),4))

Small: 1 0.3787 0.4115
Small: 2 0.3799 0.4129
Small: 3 0.38 0.414
